In [1]:
import re
import faiss
import random
import numpy as np
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datasets import Dataset
from datasets import load_dataset
from datasets import concatenate_datasets
from sentence_transformers import SentenceTransformer

c:\Users\vallu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gk_ds = load_dataset("MuskumPillerum/General-Knowledge")

In [3]:
gk_ds = gk_ds.remove_columns(['Answer'])
gk_ds= gk_ds.rename_column("Question","query")
gk_ds = gk_ds['train']
gk_ds = gk_ds.map(lambda x: {"context": ""})
gk_ds = gk_ds.map(lambda x: {"label": "1"})
gk_ds

Dataset({
    features: ['query', 'context', 'label'],
    num_rows: 37635
})

In [4]:
for row in gk_ds.shuffle(seed=42).select(range(5)):
    print(row)

{'query': 'Create a list of instructions for setting up an email account.?', 'context': '', 'label': '1'}
{'query': 'Is it ethical to use facial recognition technology for surveillance purposes?', 'context': '', 'label': '1'}
{'query': 'Suggest a reason why the US declared war on Germany in 1917.?', 'context': '', 'label': '1'}
{'query': 'What is a gripper in robotics?', 'context': '', 'label': '1'}
{'query': 'Provide 3 practice questions for multiplication?', 'context': '', 'label': '1'}


In [5]:
squad_ds = load_dataset("rajpurkar/squad")

In [6]:
squad_ds = squad_ds.remove_columns(["id", "title", "answers"])
squad_ds= squad_ds.rename_column("question","query")
squad_ds = concatenate_datasets([squad_ds['train'],squad_ds['validation']])
squad_ds = squad_ds.map(lambda x: {"label": "1"})
squad_ds

Dataset({
    features: ['context', 'query', 'label'],
    num_rows: 98169
})

In [7]:
for row in squad_ds.shuffle(seed=42).select(range(5)):
    print(row)

{'context': "Legislative power lies with the Nitijela. The upper house of Parliament, called the Council of Iroij, is an advisory body comprising twelve tribal chiefs. The executive branch consists of the President and the Presidential Cabinet, which consists of ten ministers appointed by the President with the approval of the Nitijela. The twenty-four electoral districts into which the country is divided correspond to the inhabited islands and atolls. There are currently four political parties in the Marshall Islands: Aelon̄ Kein Ad (AKA), United People's Party (UPP), Kien Eo Am (KEA) and United Democratic Party (UDP). Rule is shared by the AKA and the UDP. The following senators are in the legislative body:", 'query': 'Along with the United Democratic Party, what party currently rules the Marshall Islands?', 'label': '1'}
{'context': 'Muammar Muhammad Abu Minyar al-Gaddafi (Arabic: معمر محمد أبو منيار القذافي\u200e Arabic pronunciation: [muʕamar al.qaðaːfiː]; /ˈmoʊ.əmɑːr ɡəˈdɑːfi/;  

In [8]:
math_ds = load_dataset("HuggingFaceH4/MATH-500")
math_ds

DatasetDict({
    test: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
        num_rows: 500
    })
})

In [9]:
math_ds = math_ds.remove_columns(['solution', 'answer', 'subject', 'level', 'unique_id'])
math_ds = math_ds.rename_column("problem","query")
math_ds = math_ds.map(lambda x: {"context": ""})
math_ds = math_ds.map(lambda x: {"label": "1"})
math_ds = math_ds['test']
math_ds

Dataset({
    features: ['query', 'context', 'label'],
    num_rows: 500
})

In [10]:
for row in math_ds.shuffle(seed=42).select(range(5)):
    print(row)

{'query': 'Below is a magic square, meaning that the sum of the numbers in each row, in each column, and in each of the $2$ main diagonals are equal. What is the value of $n$?\n\n[asy]size(125);\nfor(int i = 0; i<4; ++i)\n{\n\ndraw((0,i)--(3,i),linewidth(1));\n}\n\nfor(int j = 0; j<4; ++j)\n{\n\ndraw((j,0)--(j,3),linewidth(1));\n}\n\nlabel("$n-3$",(.5,.5));\nlabel("3",(.5,1.5));\nlabel("$n+1$",(.5,2.5));\n\nlabel("$n+2$",(1.5,.5));\nlabel("$2n-9$",(1.5,1.5));\nlabel("$1$",(1.5,2.5));\n\nlabel("$2$",(2.5,.5));\nlabel("$n$",(2.5,1.5));\nlabel("$n-1$",(2.5,2.5));\n[/asy]', 'context': '', 'label': '1'}
{'query': 'Rick is thinking of a positive factor of $14$ and Steve is thinking of a positive factor of $42$.  If Rick and Steve are thinking of the same number, how many possible numbers could they be thinking of?', 'context': '', 'label': '1'}
{'query': 'At 50 miles per hour, how far would a car travel in $2\\frac{3}{4}$ hours? Express your answer as a mixed number.', 'context': '', 'label'

In [11]:
squad2_ds = load_dataset("rajpurkar/squad_v2")
squad2_ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [12]:
squad2_unanswerable_ds = squad2_ds.filter(lambda x: len(x['answers']['text']) == 0)
squad2_unanswerable_ds = concatenate_datasets([squad2_unanswerable_ds['train'], squad2_unanswerable_ds['validation']])
squad2_unanswerable_ds

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 49443
})

In [13]:
squad2_unanswerable_ds = squad2_unanswerable_ds.remove_columns(['id', 'title', 'answers'])
squad2_unanswerable_ds = squad2_unanswerable_ds.rename_column("question","query")
squad2_unanswerable_ds = squad2_unanswerable_ds.map(lambda x: {"label": "0"})
squad2_unanswerable_ds

Dataset({
    features: ['context', 'query', 'label'],
    num_rows: 49443
})

In [14]:
for row in squad2_unanswerable_ds.shuffle(seed=42).select(range(5)):
    print(row)

{'context': 'KU Endowment was established in 1891 as America’s first foundation for a public university. Its mission is to partner with donors in providing philanthropic support to build a greater University of Kansas.', 'query': 'What do donors take from the University of Kansas?', 'label': '0'}
{'context': "The independence of the ECB is instrumental in maintaining price stability. Not only must the bank not seek influence, but EU institutions and national governments are bound by the treaties to respect the ECB's independence. To offer some accountability, the ECB is bound to publish reports on its activities and has to address its annual report to the European Parliament, the European Commission, the Council of the European Union and the European Council. The European Parliament also gets to question and then issue its opinion on candidates to the executive board.", 'query': 'Why do EU institutions and national governments never respect the dependence of the ECB?', 'label': '0'}
{'

In [15]:
codeqa_ds = load_dataset('vm2825/CodeQA-dataset')
codeqa_ds

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'input_code', 'output_code'],
        num_rows: 76840
    })
})

In [16]:
codeqa_ds = codeqa_ds.remove_columns(['output_code'])
codeqa_ds = codeqa_ds.rename_column("Instruction","query")
codeqa_ds = codeqa_ds.rename_column("input_code","context")
codeqa_ds = codeqa_ds.map(lambda x: {"label": "1"})
codeqa_ds = codeqa_ds['train']
codeqa_ds

Dataset({
    features: ['query', 'context', 'label'],
    num_rows: 76840
})

In [17]:
for row in codeqa_ds.shuffle(seed=42).select(range(5)):
    print(row)

{'query': 'What adds an expose_request flag to the underlying callable ?\n', 'context': "def expose_request(func):\n\tif (not python.callable(func)):\n\t\traise TypeError('func \tmust \tbe \tcallable')\n\tif isinstance(func, types.UnboundMethodType):\n\t\tsetattr(func.im_func, '_pyamf_expose_request', True)\n\telse:\n\t\tsetattr(func, '_pyamf_expose_request', True)\n\treturn func\n", 'label': '1'}
{'query': 'What do mapped_array allocate ?\n', 'context': "@require_context\ndef mapped_array(shape, dtype=np.float, strides=None, order='C', stream=0, portable=False, wc=False):\n\t(shape, strides, dtype) = _prepare_shape_strides_dtype(shape, strides, dtype, order)\n\tbytesize = driver.memory_size_from_info(shape, strides, dtype.itemsize)\n\tbuffer = current_context().memhostalloc(bytesize, mapped=True)\n\tnpary = np.ndarray(shape=shape, strides=strides, dtype=dtype, order=order, buffer=buffer)\n\tmappedview = np.ndarray.view(npary, type=devicearray.MappedNDArray)\n\tmappedview.device_setup(

In [18]:
codealpaca_ds = load_dataset('HuggingFaceH4/CodeAlpaca_20K')
codealpaca_ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 18019
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2003
    })
})

In [19]:
codealpaca_ds = concatenate_datasets([codealpaca_ds['train'],codealpaca_ds['test']])
codealpaca_ds = codealpaca_ds.remove_columns(['completion'])
codealpaca_ds = codealpaca_ds.rename_column("prompt","query")
codealpaca_ds = codealpaca_ds.map(lambda x: {"label": "1"})
codealpaca_ds = codealpaca_ds.map(lambda x: {"context": ""})
codealpaca_ds

Dataset({
    features: ['query', 'label', 'context'],
    num_rows: 20022
})

In [20]:
for row in codealpaca_ds.shuffle(seed=42).select(range(5)):
    print(row)

{'query': 'Write a Python script to find all the numbers which are divisible by 7, but are not multiples of 5; between 2000 and 3200 (both included).\n', 'label': '1', 'context': ''}
{'query': 'Design a Java program to display the numbers from 1 to 100 in a tabular format.\n', 'label': '1', 'context': ''}
{'query': 'Write an algorithm in JavaScript to return the length of the longest word in a string.\nlet str = "The quick brown fox jumped over the lazy dog."', 'label': '1', 'context': ''}
{'query': 'Rewrite the below C statement using a ternary operator.\nif (condition) {\n    result = 5;\n}\n\nelse {\n    result = 10;\n}', 'label': '1', 'context': ''}
{'query': 'Design an algorithm in JavaScript to find maximum element within an array.\n[4,7,2,1,0,6,5]', 'label': '1', 'context': ''}


In [21]:
dolly_ds = load_dataset('databricks/databricks-dolly-15k')
dolly_ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})

In [22]:
dolly_ds = dolly_ds['train']
dolly_ds = dolly_ds.remove_columns(['response','category'])
dolly_ds = dolly_ds.rename_column("instruction","query")
dolly_ds = dolly_ds.map(lambda x: {"label": "1"})
dolly_ds

Dataset({
    features: ['query', 'context', 'label'],
    num_rows: 15011
})

In [23]:
for row in dolly_ds.shuffle(seed=42).select(range(5)):
    print(row)

{'query': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'context': '', 'label': '1'}
{'query': 'Give me a list of basic ingredients for baking cookies', 'context': '', 'label': '1'}
{'query': 'Write a funny and whimsical horoscope reading', 'context': '', 'label': '1'}
{'query': 'Who was the first person to do spacewalk?', 'context': '', 'label': '1'}
{'query': 'What causes an airplane wing to fly?', 'context': '', 'label': '1'}


In [24]:
gk_ds = gk_ds.shuffle(seed=42).select(range(10000))
squad_ds = squad_ds.shuffle(seed=42).select(range(10000))
math_ds = math_ds.shuffle(seed=42).select(range(500))
squad2_unanswerable_ds = squad2_unanswerable_ds.shuffle(seed=42).select(range(10000))
codeqa_ds = codeqa_ds.shuffle(seed=42).select(range(10000))
codealpaca_ds = codealpaca_ds.shuffle(seed=42).select(range(10000))
dolly_ds = dolly_ds.shuffle(seed=42).select(range(10000))

As of now I have decent coverage of all topics for which peopole use LLMs, now the goal is generate negatives using the data we already have.

In [25]:
datasets = [
    gk_ds,
    squad_ds,
    math_ds,
    squad2_unanswerable_ds,
    codeqa_ds,
    codealpaca_ds,
    dolly_ds
]

In [26]:
all_context = [row["context"]
    for dataset in datasets
    for row in dataset
    if row["context"].strip()
]

In [27]:
all_query = [ row["query"]
    for dataset in datasets
    for row in dataset
    if row["query"].strip()
]

In [28]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=25)
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3153.80it/s]


In [29]:
all_query_embeddings = model.encode(all_query,batch_size=16,show_progress_bar=True)

all_chunks = []
for context in all_context:
    chunks = splitter.split_text(context)
    all_chunks.extend(chunks)

all_chunk_embeddings = model.encode(all_chunks,batch_size=16,show_progress_bar=True)

query_cache = dict(zip(all_query,all_query_embeddings))
chunk_cache = dict(zip(all_chunks, all_chunk_embeddings))

Batches: 100%|██████████| 8883/8883 [22:43<00:00,  6.51it/s]


In [30]:
chunk_ids = np.arange(len(all_chunks))
chunk_data = [{'text': text, 'embedding': emb}
    for text, emb in zip(all_chunks, all_chunk_embeddings)]
chunk_store = dict(zip(chunk_ids, chunk_data))

In [31]:
def get_query_embedding(query):
    if query in query_cache:
        return query_cache[query]

    emb = model.encode(query)
    query_cache[query] = emb
    return emb


def get_chunk_embeddings(chunks):
    embeddings = []

    for chunk in chunks:
        if chunk in chunk_cache:
            embeddings.append(chunk_cache[chunk])
        else:
            emb = model.encode(chunk)
            chunk_cache[chunk] = emb
            embeddings.append(emb)

    return embeddings

In [32]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1)
    vec2 = np.asarray(vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return np.dot(vec1, vec2) / (norm1 * norm2)

def swap_context(example):
    context = random.choice(all_context)
    while context == example["context"]:
        context = random.choice(all_context)

    example["context"] = context
    example["label"]= "0"
    
    return example

def remove_context(example):
    example["context"] = ""
    example["label"]= "0"
    
    return example

def retrieve_hard_negative_chunks(index, query, k=10):

    query_emb = get_query_embedding(query)
    query_emb = np.array(query_emb).astype("float32").reshape(1, -1)
    faiss.normalize_L2(query_emb)

    scores, ids = index.search(query_emb, k)
    negatives = []

    for i in ids[0]:
        if i == -1:
            continue

        negatives.append(chunk_store[int(i)]["text"])

    return negatives

In [33]:
embeddings = np.array(all_chunk_embeddings).astype("float32")
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))

chunk_ids = np.arange(len(embeddings)).astype("int64")

index.add_with_ids(embeddings, chunk_ids)

In [34]:
def remove_best_chunk(example):
    context = example["context"]
    query = example["query"]

    chunks = splitter.split_text(context)
    if len(chunks) <= 1:
        return example

    query_emb = query_cache.get(query)
    if query_emb is None:
        query_emb = model.encode(query)
        query_cache[query] = query_emb

    chunk_embeddings = []
    for chunk in chunks:
        if chunk in chunk_cache:
            chunk_embeddings.append(chunk_cache[chunk])
        else:
            emb = model.encode(chunk)
            chunk_cache[chunk] = emb
            chunk_embeddings.append(emb)

    similarities = [cosine_similarity(query_emb, ce) for ce in chunk_embeddings]
    
    n_replace = min(2, len(chunks))
    top_idx = np.argsort(similarities)[-n_replace:][::-1]

    candidates = retrieve_hard_negative_chunks(index, query, k=50)
    valid_negatives = [c for c in candidates if c not in chunks and c != ""]

    for idx in top_idx:
        if valid_negatives:
            chunks[idx] = random.choice(valid_negatives)

    example["context"] = " ".join(chunks)
    example["label"] = "0"
    return example

Now main creating negative data

In [35]:
all_ds = concatenate_datasets(datasets)
context_required_ds = all_ds.filter(lambda x: bool(x["context"].strip()) and x["label"] == "1",num_proc=4)

def generate_negative(example):
    if random.random() < 0.6:
        return remove_best_chunk(example)
    elif random.random() < 0.5:
        return swap_context(example)
    else:
        return remove_context(example)
        

negative_ds = context_required_ds.map(generate_negative)

Map: 100%|██████████| 22955/22955 [12:09<00:00, 31.46 examples/s]


In [36]:
for row in negative_ds.shuffle(seed=42).select(range(10)):
    print(row)

{'query': 'What does f contain ?\n', 'context': '', 'label': '0'}
{'query': 'Extract the number of fights Hunt won by knockout or TKO in the below text and the year he had the most knockout wins', 'context': 'Hunt faced Roy Nelson on 20 September 2014, at UFC Fight Night 52. He won the fight via knockout in the second round. The win earned Hunt his first Performance of the Night bonus award, and the World award, and the World MMA Awards\' 2014 Knockout of the Year award. On 21 October 2014, it was announced that Hunt would replace injured UFC Heavyweight Champion Cain Velasquez in the main event of UFC 180. He faced off against Fabrício Werdum for the interim UFC for the interim UFC Heavyweight Championship. Despite having early success and dropping Werdum twice, Hunt lost the fight via TKO in the second round. Hunt faced Stipe Miocic on 10 May 2015, at UFC Fight Night 65. He lost the fight via TKO in the fifth round. Miocic set a UFC record for the most strikes landed in a fight, outl

In [37]:
final_ds = concatenate_datasets([all_ds, negative_ds])
final_ds = final_ds.shuffle(seed=42)

In [38]:
df = final_ds.to_pandas()
df.to_csv('./datasets/dataset_v1.csv', index=False)
print("Saved:", len(df), "rows")

Saved: 83455 rows
